In [1]:
import sys
sys.path.insert(0, '../lib')

import glob
import os
import pickle
import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import datasets
import numpy as np
import pandas as pd
import wandb
import scipy
import sklearn.metrics
import torch
import torch.distributions as td

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%config InlineBackend.figure_format = "retina"

In [3]:
pd.options.display.max_columns = 300
pd.options.display.max_rows = 350
pd.options.display.max_colwidth = 10000

In [4]:
mpl.rc_file_defaults()

In [5]:
TOKENS_DIR = common_data.GENEFORMER_DATA / 'tokens'
DATA_DIR = common_data.GENEFORMER_DATA / 'predictions'
N_SPLITS = 4


def load_data(token_path):
    # Filter files with prefix "chunk"
    chunks = glob.glob(f'{token_path}/chunk*.dataset')
    tokenized_chunks = [datasets.load_from_disk(file) for file in chunks]
    return datasets.concatenate_datasets(tokenized_chunks)


def filter_split_data(data, task: common_data.TaskInfo):
    task_column = task
    idx = np.isin(data[task.column], task.column_values)
    test_idx = idx & np.char.equal(data[task.split_column], 'test')
    data = data.rename_column(task.column, 'label')

    test = data.select(np.where(test_idx)[0])

    if task.column_values != [0, 1]:
        values_to_labels = {v: i for i, v in enumerate(task.column_values)}
        test = test.map(lambda x: {'label': values_to_labels[x['label']]}, num_proc=16)

    return test

In [6]:
dataset = load_data(TOKENS_DIR)

In [7]:
def get_classification_entropy(classification_probs, norm=True):
    dist = td.Categorical(probs=classification_probs)
    classification_entropy = dist.entropy()
    if norm:
        classification_entropy = classification_entropy / torch.log(input=torch.Tensor([2.]))
    return classification_entropy

In [8]:
def compute_metrics(y_pred_probs, y_pred, y_true):
    # y_pred_probs = scipy.special.softmax(logits, axis=1)
    # y_pred = np.argmax(y_pred_probs, axis=-1)

    auprc = sklearn.metrics.average_precision_score(y_true, y_pred_probs[:, 1])
    f1_binary = sklearn.metrics.f1_score(y_true, y_pred, average='binary')
    roc_auc = sklearn.metrics.roc_auc_score(y_true, y_pred_probs[:, 1])
    accuracy = sklearn.metrics.accuracy_score(y_true, y_pred)
    recall = sklearn.metrics.recall_score(y_true, y_pred)
    precision = sklearn.metrics.precision_score(y_true, y_pred)

    return {
        'accuracy': accuracy,
        'f1_binary': f1_binary,
        'AUROC': roc_auc,
        'AUPRC': auprc,
        'recall': recall,
        'precision': precision,
    }

### Show data types and structure before processing everything

Processing everything takes ~1.5-2h, so I did it once and saved as pickle

Files loaded here are generated by this script: `scripts/run_predict.py` and this script: `scripts/save_avg_mc_probs.py` via the Snakefile

Function `filter_split_data` above also comes from that script, so we have the same logic of getting the test data in this notebook.

In [ ]:
for task_path in common_data.get_tasks(context='geneformer'):
    for split_path in [f'split_{i + 1}' for i in range(N_SPLITS)]:
        if not (DATA_DIR / task_path / split_path).exists():
            continue
        task_info = common_data.get_task_info(task_path, split_path)
        test_data = filter_split_data(dataset, task_info)
        for model_path in (DATA_DIR / task_path / split_path).iterdir():
            if not model_path.name.startswith('model_'):
                continue
            if not (model_path / 'test_predictions/avg_probs.npy').exists():
                continue

            print(f'Loading data for model {model_path}')
            cell_ids = np.load(model_path / 'test_predictions/cell_ids.npy')
            print(f'Cell ids (custom order of test cells) shape {cell_ids.shape}, first value: {cell_ids[0]}')

            # Create a dictionary mapping cell_ids to their indices
            cell_id_to_index = {cell_id: index for index, cell_id in enumerate(cell_ids)}
            test_data = test_data.map(lambda x: {"sort_index": cell_id_to_index[x['obs_names']]})
            test_data = test_data.sort("sort_index")
            test_data = test_data.remove_columns("sort_index")
            print(f'Test data shape {len(test_data)}, first cell_id {test_data["obs_names"][0]}')

            probs = np.load(model_path / 'test_predictions/avg_probs.npy')
            print(f'Loaded probabilities shape {probs.shape}, all sum to 1: {np.allclose(probs.sum(axis=1), 1)}')
            preds = np.argmax(probs, axis=-1)

            info = compute_metrics(probs, preds, test_data['label'])
            info['task'] = task_path
            info['split'] = split_path
            info['model'] = model_path.name
            print('Model metrics:')
            print(info)
            break
        break
    break

Loading data for model /gpfs/projects/b1196/ewa_group/serniczek/08_geneformer/../data/08_geneformer/predictions/is_episode_cured/split_1/model_2
Cell ids (custom order of test cells) shape (68740,), first value: SC410_ACTTACTGTGTGAATA
Test data shape 68740, first cell_id SC410_ACTTACTGTGTGAATA
Loaded probabilities shape (68740, 2), all sum to 1: True
Model metrics:
{'accuracy': 0.5165696828629619, 'f1_binary': 0.4914063576118398, 'AUROC': 0.5323055832074071, 'AUPRC': 0.5788840967050577, 'recall': 0.42057005134653674, 'precision': 0.5909375345087791, 'task': 'is_episode_cured', 'split': 'split_1', 'model': 'model_2'}


In [ ]:
%%time
# Use for normal

df = []
model_perf = {}

for task_path in common_data.get_tasks(context='geneformer'):
    for split_path in [f'split_{i + 1}' for i in range(N_SPLITS)]:
        if not (DATA_DIR / task_path / split_path).exists():
            continue
        task_info = common_data.get_task_info(task_path, split_path)
        test_data = filter_split_data(dataset, task_info)
        for model_path in (DATA_DIR / task_path / split_path).iterdir():
            if not model_path.name.startswith('model_'):
                continue
            if not (model_path / 'test_predictions/avg_probs.npy').exists():
                continue

            cell_ids = np.load(model_path / 'test_predictions/cell_ids.npy')

            # Create a dictionary mapping cell_ids to their indices
            cell_id_to_index = {cell_id: index for index, cell_id in enumerate(cell_ids)}
            test_data = test_data.map(lambda x: {"sort_index": cell_id_to_index[x['obs_names']]})
            test_data = test_data.sort("sort_index")
            test_data = test_data.remove_columns("sort_index")

            probs = np.load(model_path / 'test_predictions/avg_probs.npy')
            preds = np.argmax(probs, axis=-1)
            entropy = get_classification_entropy(torch.tensor(probs))
            info = compute_metrics(probs, preds, test_data['label'])
            info['task'] = task_path
            info['split'] = split_path
            info['model'] = model_path.name
            df.append(info)
            model_df = pd.DataFrame(dict(
                true_label=test_data['label'],
                cell_types=test_data['Level_6'],
                obs_names=test_data['obs_names'],
                sample=test_data['individual'],
                pred_label=preds,
                entropy=entropy,
                pos_class_prob=probs[:, 1]
            ))
            model_perf[str(model_path)] = model_df
            print('.', flush=True, end='')
    print('', flush=True)
    print(task_path + ' done')

....................
is_episode_cured done


Map: 100%|██████████| 183290/183290 [00:28<00:00, 6467.72 examples/s]


.

Map: 100%|██████████| 183290/183290 [00:29<00:00, 6284.34 examples/s]


.

...

.

Map: 100%|██████████| 83644/83644 [00:11<00:00, 7301.72 examples/s]


.

Map: 100%|██████████| 83644/83644 [00:11<00:00, 7294.35 examples/s]


.
N_vs_G-_G+ done
CPU times: user 1h 49min 55s, sys: 2min, total: 1h 51min 55s
Wall time: 1h 53min 9s


In [42]:
df = pd.DataFrame(df)

In [44]:
with open('02_model_perf.pkl', 'wb') as f:
    pickle.dump(model_perf, f)

with open('02_model_df.pkl', 'wb') as f:
    pickle.dump(df, f)